# Pretrain a small language model with TrainLM

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/Dhiraj309/TrainLM/main/notebooks/TrainLM_Native_TorchTPU_From_Scratch.ipynb)

Open a copy in Colab or import it into Kaggle. Choose a single-device native TorchTPU runtime for this recipe; the link does not provision a TPU.

## What this runs

A single-device native TorchTPU run using a Hugging Face GPT-2 config/tokenizer and raw TinyStories text. `MODEL_INITIALIZATION="random"` starts with random weights without downloading model weights; set it to `"pretrained"` for CPT from GPT-2 weights. The default `max_tokens=10_000` (10,000-token) budget is a workflow smoke test; it does not produce a useful model or certify TPU performance. Increase `max_tokens` and `max_train_examples` for a longer experiment. The run evaluates on a held-out split and exports a model that Transformers loads in a clean CPU process.

## Choose a recipe

| Goal | Model initialization | Data |
| --- | --- | --- |
| Scratch | `"random"` | Keep the TinyStories text defaults. |
| Continued pretraining | `"pretrained"` | Use the same raw corpus settings or choose your text dataset. |
| Pretraining on token IDs | Either | Set `DATA_FORMAT="tokens"`, select your tokenized dataset ID and splits, and choose the model whose tokenizer produced those IDs. |

Choose a fresh run directory for each recipe. Each run evaluates on the validation split and writes a standard Transformers export. After the artifact cell, use the optional Hub cell to upload the selected run to a private repository.

## Run and resume

Run the setup cell, then the fresh-run cell. It prints a pre-run plan before training. For checkpoint recovery after an interruption, use the resume cell and choose `RESUME_MODE="resume"` for exact resume or `"extend"` for a larger token budget. It prints the lifecycle plan before it resumes. Do not rerun setup or fresh training to resume. For a new independent scratch/CPT run, select the model initialization and use a new `RUN_DIR`. After training, run the artifact cell to preview and download the report/export bundle. The final optional cell can publish the selected run privately to Hugging Face; it is disabled by default. This notebook targets a single native TPU device; it does not certify multi-device execution.

For configuration details and limitations, see the [pretraining guide](../README.md#start-here-pretrain-a-language-model-from-scratch).


In [ ]:
import subprocess
import sys

source = "trainlm @ git+https://github.com/Dhiraj309/TrainLM.git@main"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", source],
    check=True,
)
requirements = [
    "huggingface-hub>=1.5,<2",
    "PyYAML>=6,<7",
    "transformers>=5.15,<6",
    "datasets>=3,<5",
    "tokenizers>=0.21,<1",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *requirements],
    check=True,
)
print("TrainLM installed without asking pip to replace PyTorch or TorchTPU.")
from importlib.metadata import PackageNotFoundError, version

for package_name in ("torch-tpu", "libtpu"):
    try:
        print(f"{package_name}=={version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name}=MISSING; native TPU needs a runtime that provides TorchTPU")


In [ ]:
import json
from importlib.metadata import distribution
from trainlm import TrainLMPretrainingConfig, run_pretraining
source = json.loads(distribution("trainlm").read_text("direct_url.json") or "{}")
source_revision = source.get("vcs_info", {}).get("commit_id")
assert source_revision, "Reinstall TrainLM from Git to record its exact commit."
RUN_DIR = "trainlm-pretraining-fresh"
FRESH_MAX_TOKENS, FRESH_MAX_TRAIN_EXAMPLES, LOGGING_STEPS = 10_000, 2_000, 10
MODEL_ID, MODEL_INITIALIZATION = "openai-community/gpt2", "random"
DATASET_ID, DATASET_CONFIG_NAME, DATASET_REVISION = "roneneldan/TinyStories", None, "main"
TRAIN_SPLIT, VALIDATION_SPLIT = "train", "validation"
DATA_FORMAT = "text"  # "tokens" for input_ids encoded with the model tokenizer.
config = TrainLMPretrainingConfig(
    output_dir=RUN_DIR,
    max_tokens=FRESH_MAX_TOKENS, max_train_examples=FRESH_MAX_TRAIN_EXAMPLES,
    sequence_length=128, logging_steps=LOGGING_STEPS,
    model_name_or_path=MODEL_ID, model_initialization=MODEL_INITIALIZATION,
    dataset_id=DATASET_ID, dataset_config_name=DATASET_CONFIG_NAME,
    dataset_revision=DATASET_REVISION, train_split=TRAIN_SPLIT,
    validation_split=VALIDATION_SPLIT, data_format=DATA_FORMAT, text_field="text", accelerator="native_tpu",
)
print(json.dumps(config.training_plan(inspect_model=True), indent=2))
result = run_pretraining(config, show_progress=True, source_revision=source_revision)
print(result.summary())
print(f"checkpoint: {result.checkpoint_path} | report: {result.report_path} | model: {result.export_path}")


In [ ]:
import json
from importlib.metadata import distribution
from dataclasses import replace
from trainlm import restore_pretraining_report, run_pretraining
source = json.loads(distribution("trainlm").read_text("direct_url.json"))
source_revision = source["vcs_info"]["commit_id"]
RUN_DIR, RESUME_MODE = "trainlm-pretraining-fresh", "skip"  # "resume" or "extend"
if RESUME_MODE in {"resume", "extend"}:
    saved = restore_pretraining_report(f"{RUN_DIR}/report.json")
    resume_config = saved.config
    extend_budget = RESUME_MODE == "extend"
    if extend_budget:
        extra_tokens = max(2_000, resume_config.max_tokens // 5)
        resume_config = replace(resume_config, max_tokens=resume_config.max_tokens + extra_tokens)
    options = {"resume_from_checkpoint": saved.checkpoint_path, "extend_training_budget": extend_budget}
    print(json.dumps(resume_config.training_plan(**options), indent=2))
    resumed = run_pretraining(
        resume_config, **options, show_progress=True, source_revision=source_revision,
        report_path=f"{RUN_DIR}/resume-report.json", export_path=f"{RUN_DIR}/resumed-export",
    )
    print(resumed.summary())
    print(f"Resumed from: {saved.checkpoint_path} | report: {resumed.report_path} | model: {resumed.export_path}")
else:
    print("Choose RESUME_MODE='resume' or 'extend', then run this cell only.")


In [ ]:
from trainlm import bundle_pretraining_artifacts
RUN_RESUME = globals().get("RESUME_MODE", "skip") in {"resume", "extend"}
run = resumed if RUN_RESUME else result
model_path = getattr(run, "export_path", None)
if model_path:
    import subprocess, sys
    inference_code = '''
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
model_path = sys.argv[1]
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)
model.eval()
inputs = tokenizer("Once upon a time", return_tensors="pt")
with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=24, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))
'''
    subprocess.run([sys.executable, "-I", "-c", inference_code, str(model_path)], check=True)
if RUN_RESUME:
    bundle = bundle_pretraining_artifacts(resumed.report_path)
else:
    bundle = bundle_pretraining_artifacts(result.report_path)
try:
    from google.colab import files
except ImportError:
    print(f"Artifact bundle: {bundle}")
else:
    files.download(str(bundle))


## Optional: publish a private Hub copy

This cell is off by default. Set a Hugging Face write token through `hf auth login` or the notebook login prompt, then set your repository ID. The upload remains private and prints only the repository URL and commit SHA.

Use this after running either the fresh-run or resume cell. TrainLM uploads only the model export and model card; it does not upload the run report, corpus, or resumable checkpoints.


In [ ]:
PUBLISH_TO_HUB = False
HF_REPO_ID = "your-hf-username/trainlm-private-smoke"

if PUBLISH_TO_HUB:
    from huggingface_hub import get_token, login

    if get_token() is None:
        login(add_to_git_credential=False)
    published = run.push_to_hub(
        HF_REPO_ID,
        private=True,
        commit_message="Publish TrainLM private pretraining smoke",
    )
    print(f"repository={published.repo_url}")
    print(f"commit={published.revision}")
else:
    print("Hub publishing skipped; set PUBLISH_TO_HUB = True to opt in.")
